# Top OWID Charts of 2025

This notebook replicates the top charts from [Our World in Data's Top Charts of 2025](https://ourworldindata.org/top-of-the-charts-2025) using the `owid-grapher-py` library.

In [1]:
# Install the package if not already available
try:
    import owid.grapher
except ImportError:
    %pip install owid-grapher-py

try:
    %load_ext autoreload
    %autoreload 2
except:
    pass  # autoreload not available (e.g., in Google Colab with Python 3.12+)

import pandas as pd
from owid.grapher import Chart

## 1. Annual CO₂ Emissions per Country

Source: https://ourworldindata.org/grapher/annual-co2-emissions-per-country

In [2]:
# Fetch data
df_co2 = pd.read_csv(
    "https://ourworldindata.org/grapher/annual-co2-emissions-per-country.csv?useColumnShortNames=true"
)
df_co2 = df_co2.rename(columns={"Entity": "entity", "Year": "year"})
df_co2.head()

,entity,Code,year,emissions_total
0,Afghanistan,AFG,1949,14656.0
1,Afghanistan,AFG,1950,84272.0
2,Afghanistan,AFG,1951,91600.0
3,Afghanistan,AFG,1952,91600.0
4,Afghanistan,AFG,1953,106256.0


In [3]:
Chart(df_co2).mark_line().mark_bar().mark_map(
    color_scheme="Reds",
    binning_strategy="manual",
    custom_numeric_values=[0, 3e6, 1e7, 3e7, 1e8, 3e8, 1e9, 3e9, 1e10],
).encode(
    x="year",
    y="emissions_total",
    entity="entity",
).yaxis(
    unit="t",
).label(
    title="Annual CO₂ emissions",
    subtitle="Carbon dioxide emissions from fossil fuels and industry, excluding land-use change emissions.",
    source_desc="Global Carbon Budget (2024)",
).select(
    entities=["China", "United States", "India", "Germany", "France", "United Kingdom", "Brazil"],
).interact(
    allow_relative=True,
    entity_control=True,
)

## 2. Life Expectancy

Source: https://ourworldindata.org/grapher/life-expectancy

In [4]:
# Fetch data
df_life = pd.read_csv(
    "https://ourworldindata.org/grapher/life-expectancy.csv?useColumnShortNames=true"
)
df_life = df_life.rename(columns={"Entity": "entity", "Year": "year"})
df_life.head()

,entity,Code,year,life_expectancy_0
0,Afghanistan,AFG,1950,28.1563
1,Afghanistan,AFG,1951,28.5836
2,Afghanistan,AFG,1952,29.0138
3,Afghanistan,AFG,1953,29.4521
4,Afghanistan,AFG,1954,29.6975


In [5]:
Chart(df_life).mark_line().mark_bar().mark_map(
    color_scheme="YlGnBu",
    binning_strategy="manual",
    custom_numeric_values=[40, 45, 50, 55, 60, 65, 70, 75, 80, 85],
    time_tolerance=10,
).encode(
    x="year",
    y="life_expectancy_0",
    entity="entity",
).yaxis(
    unit="years",
).label(
    title="Life expectancy",
    subtitle="Period life expectancy at birth, in years.",
    source_desc="UN WPP (2024); HMD (2024); Zijdeman et al. (2015); Riley (2005)",
).select(
    entities=["World", "Americas", "Europe", "Africa", "Asia", "Oceania"],
).interact(
    entity_control=True,
)

## 3. Democracy Index

Source: https://ourworldindata.org/grapher/democracy-index-eiu

In [6]:
# Fetch data
df_democracy = pd.read_csv(
    "https://ourworldindata.org/grapher/democracy-index-eiu.csv?useColumnShortNames=true"
)
df_democracy = df_democracy.rename(columns={"Entity": "entity", "Year": "year"})

# CSV has region info only for last year, propagate it to all rows
df_democracy['owid_region'] = df_democracy.entity.map(
    df_democracy.dropna(subset='owid_region').set_index('entity')['owid_region']
).values


df_democracy.head()

,entity,Code,year,democracy_eiu,owid_region
0,Afghanistan,AFG,2006,3.06,Asia
1,Afghanistan,AFG,2008,3.02,Asia
2,Afghanistan,AFG,2010,2.48,Asia
3,Afghanistan,AFG,2011,2.48,Asia
4,Afghanistan,AFG,2012,2.48,Asia


In [7]:
# Note: Original uses custom colors (red-to-blue gradient). We use RdYlBu as closest built-in scheme.
# Original chartTypes: LineChart, SlopeChart, DiscreteBar, Marimekko
Chart(df_democracy).mark_line().mark_slope().mark_bar().mark_marimekko().mark_map(
    color_scheme="RdYlBu",
    binning_strategy="manual",
    custom_numeric_values=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
).show("map").encode(
    x="year",
    y="democracy_eiu",
    entity="entity",
    color="owid_region",
).label(
    title="Democracy Index",
    subtitle="Expert estimates of the extent to which citizens can choose their political leaders in free and fair elections, enjoy civil liberties, prefer democracy over other political systems, can and do participate in politics, and have a functioning government that acts on their behalf. The index ranges from 0 to 10 (most democratic).",
    source_desc="Economist Intelligence Unit",
).select(
    entities=["Argentina", "Australia", "Botswana", "China"],
).interact(
    entity_control=True,
)

## 4. Temperature Anomaly

Source: https://ourworldindata.org/grapher/temperature-anomaly

This chart demonstrates **confidence intervals** - the shaded area shows the uncertainty range around the main temperature estimate.

In [8]:
# Fetch data
df_temp = pd.read_csv(
    "https://ourworldindata.org/grapher/temperature-anomaly.csv?useColumnShortNames=true"
)
df_temp = df_temp.rename(columns={"Entity": "entity", "Year": "year"})
df_temp.head()

,entity,Code,year,near_surface_temperature_anomaly,near_surface_temperature_anomaly_lower,near_surface_temperature_anomaly_upper
0,Northern Hemisphere,NaN,1850,-0.055067,-0.254982,0.144848
1,Northern Hemisphere,NaN,1851,0.161580,-0.047569,0.370729
2,Northern Hemisphere,NaN,1852,0.145127,-0.076860,0.367114
3,Northern Hemisphere,NaN,1853,0.135437,-0.082370,0.353244
4,Northern Hemisphere,NaN,1854,0.206140,0.009939,0.402340


In [ ]:
# Line chart with confidence intervals (shaded area for uncertainty)
# Use entity_mode="change-country" to restrict to single entity (required for CI colors)
Chart(df_temp).mark_line().mark_bar().encode(
    x="year",
    y="near_surface_temperature_anomaly",
    y_lower="near_surface_temperature_anomaly_lower",
    y_upper="near_surface_temperature_anomaly_upper",
    entity="entity",
).variable(
    "near_surface_temperature_anomaly",
    name="Average",
    color="#ca2628",
).variable(
    "near_surface_temperature_anomaly_lower",
    name="Lower bound (95% CI)",
    color="#c8c8c8",
).variable(
    "near_surface_temperature_anomaly_upper",
    name="Upper bound (95% CI)",
    color="#c8c8c8",
).yaxis(
    unit="°C",
).label(
    title="Annual temperature anomalies relative to the pre-industrial period",
    subtitle="The difference in average land-sea surface temperature compared to the 1861-1890 mean, in degrees Celsius.",
    source_desc="Met Office Hadley Centre",
).select(
    entities=["World"],
).interact(
    entity_mode="change-country",
)

## 5. GDP per Capita

Source: https://ourworldindata.org/grapher/gdp-per-capita-worldbank

In [10]:
# Fetch data
df_gdp = pd.read_csv(
    "https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?useColumnShortNames=true"
)
df_gdp = df_gdp.rename(columns={"Entity": "entity", "Year": "year"})
df_gdp.head()

,entity,Code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,NaN
1,Afghanistan,AFG,2001,1454.1108,NaN
2,Afghanistan,AFG,2002,1774.3087,NaN
3,Afghanistan,AFG,2003,1815.9282,NaN
4,Afghanistan,AFG,2004,1776.9182,NaN


In [11]:
# Original chartTypes: LineChart, SlopeChart, DiscreteBar, Marimekko
Chart(df_gdp).mark_line().mark_slope().mark_bar().mark_marimekko().mark_map(
    color_scheme="GnBu",
    binning_strategy="manual",
    custom_numeric_values=[0, 1000, 2000, 5000, 10000, 20000, 50000, 100000],
).show("map").encode(
    x="year",
    y="ny_gdp_pcap_pp_kd",
    entity="entity",
).yaxis(
    unit="$",
    scale_control=True,
).label(
    title="GDP per capita",
    subtitle="GDP per capita is a country's gross domestic product divided by its population. This data is adjusted for inflation and differences in living costs between countries.",
    source_desc="Eurostat, OECD, IMF, and World Bank (2025)",
).select(
    entities=["United States", "Germany", "United Kingdom", "Brazil", "South Korea", "Japan", "China", "India", "Ethiopia", "Russia"],
).interact(
    entity_control=True,
)